# 9.10 高级投机解码 (EAGLE / MTP)

> 🕐 预估学习时间：35分钟

在基础投机解码之上，EAGLE 用特征层草稿、Medusa/MTP 用多头并行猜 token。本节实现接受率统计与树状验证的教学版本。

本节涵盖：
- 草稿-验证接受率
- 特征级草稿（EAGLE 直觉）
- 多 token 预测头（MTP/Medusa）
- 树状候选与期望加速比


## 1. 经典投机解码接受率

草稿模型提 γ 个 token，目标模型一次前向验证；按概率比接受/拒绝。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


class TinyPolicy(nn.Module):
    def __init__(self, vocab=32, d=32):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.head = nn.Linear(d, vocab)

    def forward(self, x):
        h, _ = self.rnn(self.embed(x))
        return self.head(h)


draft = TinyPolicy()
target = TinyPolicy()
# make target sharper copy-ish
target.load_state_dict(draft.state_dict())


def speculative_step(prefix, gamma=4):
    # draft autoregressive proposals
    seq = prefix
    draft_tokens = []
    draft_probs = []
    with torch.no_grad():
        for _ in range(gamma):
            logits = draft(seq)[:, -1]
            probs = F.softmax(logits, dim=-1)
            tok = torch.multinomial(probs, 1)
            draft_tokens.append(tok)
            draft_probs.append(probs.gather(-1, tok))
            seq = torch.cat([seq, tok], dim=1)
        # target verifies all prefixes in one forward
        full = torch.cat([prefix] + draft_tokens, dim=1)
        t_logits = target(full[:, :-1])
        t_probs = F.softmax(t_logits[:, -gamma:], dim=-1)
        accepted = []
        for i, tok in enumerate(draft_tokens):
            p_t = t_probs[:, i, :].gather(-1, tok)
            p_d = draft_probs[i]
            ratio = (p_t / (p_d + 1e-12)).clamp(max=1.0)
            if torch.rand(()) <= ratio:
                accepted.append(tok)
            else:
                # sample from residual distribution (simplified: sample target)
                tok_new = torch.multinomial(t_probs[:, i, :], 1)
                accepted.append(tok_new)
                break
        else:
            # bonus sample from target at next position
            pass
    return torch.cat(accepted, dim=1), len(accepted)


prefix = torch.randint(0, 32, (1, 5))
got, n_acc = speculative_step(prefix, gamma=4)
print('=== Speculative Decoding ===')
print(f'accepted_tokens={n_acc}, values={got.tolist()}')
print(f'Key: Speedup ≈ accepted_length / (1 + draft_cost/target_cost); acceptance rate is everything.')


## 2. EAGLE 直觉：用特征而不是 token 喂草稿

EAGLE 草稿头读取目标模型倒数层特征，再预测下一 token，使草稿分布更接近目标，提高接受率。


In [ ]:
class EagleLike(nn.Module):
    def __init__(self, vocab=32, d=32):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.draft_head = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, vocab))
        self.lm_head = nn.Linear(d, vocab)

    def features(self, x):
        h, _ = self.rnn(self.embed(x))
        return h

    def forward(self, x):
        h = self.features(x)
        return self.lm_head(h), self.draft_head(h)


eagle = EagleLike()
opt = torch.optim.Adam(eagle.parameters(), lr=2e-3)
print('=== Train Eagle-like Draft Head ===')
for step in range(40):
    x = torch.randint(0, 32, (16, 12))
    logits, draft_logits = eagle(x)
    # draft predicts next token from current features
    loss_lm = F.cross_entropy(logits[:, :-1].reshape(-1, 32), x[:, 1:].reshape(-1))
    loss_draft = F.cross_entropy(draft_logits[:, :-1].reshape(-1, 32), x[:, 1:].reshape(-1))
    loss = loss_lm + loss_draft
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 10 == 0 or step == 39:
        with torch.no_grad():
            agree = (draft_logits[:, :-1].argmax(-1) == logits[:, :-1].argmax(-1)).float().mean()
        print(f'step={step:02d} loss={loss.item():.4f} draft_target_agree={agree.item():.3f}')
print(f'\nKey: Feature-conditioned draft heads track the target distribution more closely than a tiny separate LM.')


## 3. MTP / Medusa：多头并行猜测

多个头分别预测 +1/+2/+3… 位置，一次前向产生短树候选，再由主干验证。


In [ ]:
class MedusaHeads(nn.Module):
    def __init__(self, vocab=32, d=32, n_heads=3):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.rnn = nn.GRU(d, d, batch_first=True)
        self.base = nn.Linear(d, vocab)
        self.extra = nn.ModuleList([nn.Linear(d, vocab) for _ in range(n_heads)])

    def forward(self, x):
        h, _ = self.rnn(self.embed(x))
        outs = [self.base(h)]
        for head in self.extra:
            outs.append(head(h))
        return outs  # list of logits for offset 0..n


medusa = MedusaHeads()
x = torch.randint(0, 32, (8, 10))
outs = medusa(x)
print('=== Medusa / MTP Heads ===')
for i, logit in enumerate(outs):
    print(f'head+{i}: {tuple(logit.shape)}')

# Expected tokens accepted under independent accept p
def expected_accept_len(p, gamma):
    # 1 + p + p^2 + ... until rejection geometry for a chain
    return (1 - p ** (gamma + 1)) / (1 - p) if p < 1 else gamma + 1

print(f'\nExpected accepted length if p_accept=0.7, gamma=3: {expected_accept_len(0.7, 3):.2f}')
print(f'Key: Tree/MTP candidates raise expected accepted tokens per expensive target forward.')


## 课后思考题

1. 接受率从 0.5 提到 0.8 对端到端加速比意味着什么？
2. EAGLE 草稿依赖目标特征，部署时如何与 CUDA Graph / PD 分离兼容？
3. MTP 训练是否损害下一 token 质量？如何加权多头损失？
4. 何时该用小模型投机，何时该用同模型多头？

---
> 本节涵盖了9.10 高级投机解码的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
